# NIT-4 · 5 — The national plan

**The failure this notebook prevents.** The four final reports each give a number for
their own plans, in their own cities, over their own years. The plan on Congress's
table — `G = 0.75, t = 0.50`, for every eligible family in the country, indefinitely —
was run nowhere in exactly that form. The tempting move is to pick the site whose mean
plan is closest, quote its contrast, and call it the answer. That number would be
Rural's, which is the smallest experiment, in the two most rural states, with the
strongest response of the four.

The honest answer is a **surface**: the hours change as a function of the plan,
fitted to every family the licensed experiments enrolled, realized at the plan on the
table with its interval, and then carried — with the assumptions that carrying takes,
named — from the families the experiments enrolled to the families a national plan
would reach.

| | reaches into |
|---|---|
| the surface, and the surface the data refuses | `surface.SurfaceSpec`, `surface.fit`, `core.Unverified` |
| the contrast the decision needs | `estimands.realize`, `estimands.EstimandResult`, `identify.identify` |
| the curve and where it turns | `surface.marginal_band` |
| from the enrolled to the national population | `Estimand.transfer_to`, `identify.transport_verdict`, `identify.selection_diagram` |
| the decision, and what would change it | `design.DecisionSpec`, `design.evpi_gaussian`, `design.time_to_re_experiment` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd

import guarantee as W
from axiom.core import D, Unverified, is_failure
from axiom.design import DecisionSpec, evpi_gaussian, time_to_re_experiment
from axiom.estimands import EstimandResult, realize
from axiom.identify import CausalGraph, identify, selection_diagram, transport_verdict
from axiom.surface import FitResult, fit, marginal_band
from axiom.viz import causal_graph
from axiom.display import enable, show, table

from _style import caption, compare, curve_band, heat, intervals, lines, mark_x, mark_y

enable();  # every axiom result renders itself from here on

programme = W.run_programme()
G_NAT, T_NAT = W.NATIONAL_PLAN
LICENSED = ("rural", "gary", "seattle_denver")  # notebook 3: New Jersey's readout answers another reference

## 1 · The families the surface may use

Notebook 3's conformance table said which readouts answer the estimands their plans
committed to. New Jersey's does not — its reference moved — so its families do not
enter the surface, and the three-year cohorts of the other three sites do. The
families are laid out as a balanced panel of blocks, because the surface wants
`(unit, time)` and a cross-section has neither.

In [ ]:
panel = W.analysis_panel(programme, sites=LICENSED, block=100)
print(panel)
frame = panel.frame
print(f"{len(frame):,} families on {frame[['guarantee', 'tax_rate']].drop_duplicates().shape[0]} distinct plans, "
      f"{(frame.guarantee == 0).mean():.0%} of them controls")

## 2 · The surface the data refuses, and the one it allows

Notebook 2 warned that the sites' interior designs could see a curvature in the tax
rate and the corners could not. Try it: a quadratic in the tax rate, fitted by Laplace.
The fit comes back `Unverified`, with the reason — the posterior is flat in one
direction — because the tax rates the three sites actually ran do not pin a parabola
apart from its slope once a prior this wide is allowed. That is a typed refusal where
a sampler would have returned a number and a warning nobody read.

The linear surface with an interaction is what the data supports, and it is the
planning surface of notebook 2.

In [ ]:
quadratic = fit(W.analysis_spec(), panel, backend="laplace", draws=1000, seed=0)
print("quadratic in the tax rate:", type(quadratic.posterior).__name__)
print("  ", quadratic.posterior.reason if isinstance(quadratic.posterior, Unverified) else "")

linear: FitResult = fit(W.spec(name="nit4_linear"), panel, backend="laplace", draws=1500, seed=0)
print("\nlinear with interaction: converged", linear.converged)
rows = []
for name in ("alpha", "beta1_guarantee", "beta1_tax_rate", "gamma_guarantee_tax_rate", "sigma"):
    s = linear.posterior.summary(name, mass=0.9)
    rows.append([name, f"{s.mean:.1f}", f"{s.sd:.1f}", s.interval.text()])
table(rows, headers=("parameter", "posterior mean", "sd", "90% HDI"),
      title="the surface, in hours per unit of scaled dose (u = dose / top of the lattice)")

## 3 · The contrast the decision needs

`realize` evaluates notebook 1's estimand against the fit and refuses unless the
identification verdict permits it. The verdict is the randomized graph's — nothing
to adjust for — and the result carries its status, its ledger and its interval
definition. Then the same estimand at the other plans Congress has discussed, from
the same posterior.

In [ ]:
randomized = CausalGraph.from_edges(
    "taste -> hours, taste -> schooling, tax_rate -> hours, guarantee -> hours",
    unmeasured=["taste"], name="what_the_experiments_know",
)
verdict = identify(randomized, "guarantee", "hours")
contrast = W.national_estimand()
answer = realize(contrast, linear, verdict=verdict.verdict, mass=0.9, definition="hdi")
assert isinstance(answer, EstimandResult)
print(f"{contrast.name}: {answer.summary.mean:.0f} {answer.unit}, {answer.summary.interval.text()}")
print(f"status {answer.status} | assumptions {[a.name for a in answer.assumptions] or 'none'}")
print("  (the estimand names a unit for one treatment; the other's level is taken in the producer's, and the result says so)")
print(f"the truth at a typical site (synthetic world): {float(W.hours_change(G_NAT, T_NAT)):.0f}")
print(f"the Treasury's threshold: {-W.HOURS_THRESHOLD:.0f}")

rows, forest = [], []
for g, t in ((0.5, 0.5), (0.75, 0.5), (0.75, 0.7), (1.0, 0.5), (1.0, 0.7)):
    r = realize(W.national_estimand(plan=(g, t), name=f"plan_{g}_{t}"), linear, verdict=verdict.verdict, mass=0.9, definition="hdi")
    assert isinstance(r, EstimandResult)
    truth = float(W.hours_change(g, t))
    rows.append([W.plan_label(g, t), f"{r.summary.mean:.0f}", r.summary.interval.text(), f"{truth:.0f}",
                 f"{W.plan_cost(g, t) * W.N_ELIGIBLE / 1e9:.1f} bn"])
    forest.append((W.plan_label(g, t), r.summary.mean, r.summary.interval.lower, r.summary.interval.upper))
table(rows, headers=("plan", "hours change", "90% HDI", "truth (synthetic world)", "programme cost/yr"))
fig = intervals(
    forest, ref=-W.HOURS_THRESHOLD, ref_label="the Treasury's threshold", highlight=W.plan_label(G_NAT, T_NAT),
    title="The same surface, at the plans Congress has discussed",
    subtitle="hours change per family head, three-year enrolment, with 90% HDIs",
    x_title=f"hours change ({W.OUTCOME_UNIT})",
)
caption(fig, "The plan on the table comes in under the threshold, with room. A tax rate of 70 % "
             "does not, at either guarantee; the cheaper plan costs more hours, which is the "
             "trade the two doses exist to describe and the reason a single number would not "
             "have done.")

In [ ]:
grid_g, grid_t = W.GUARANTEES, W.TAX_RATES
means = np.array([[realize(W.national_estimand(plan=(g, t), name="grid"), linear, verdict=verdict.verdict,
                           mass=0.9, definition="hdi").summary.mean for t in grid_t] for g in grid_g])
fig = heat(
    means, [f"t = {t}" for t in grid_t], [f"G = {g}" for g in grid_g],
    colorbar_title=W.OUTCOME_UNIT, text_fmt="{:.0f}",
    title="The labour-supply surface",
    subtitle="posterior mean of the hours change at every plan on the lattice",
    x_title="tax rate", y_title="guarantee",
)
caption(fig, "Down and to the right is more hours lost. The guarantee and the tax rate each cost "
             "hours, and together they cost more than the sum — the interaction the surface "
             "carries and the survey of notebook 1 could not have seen.")

## 4 · Where the next point of tax rate stops paying

`marginal_band` is the derivative of the surface in one dose with its interval. In
the tax rate, at the guarantee on the table, it is the number a negotiator needs:
how many hours each further point of tax rate costs, and whether the data knows.

In [ ]:
band = marginal_band(linear, "tax_rate", n_grid=17, mass=0.9)
assert not is_failure(band)
per_point = np.asarray(band.mean) / 100.0  # per percentage point of tax rate
fig = curve_band(
    band.doses, per_point, np.asarray(band.lower) / 100.0, np.asarray(band.upper) / 100.0,
    label="marginal cost of a point of tax rate",
    title="What each further point of tax rate costs",
    subtitle=f"d(hours change) / d(tax rate) per percentage point, {band.label()}",
    x_title="tax rate", y_title="hours per point of tax rate",
)
mark_x(fig, T_NAT, text="the plan on the table")
print(f"at t = {T_NAT}: {float(np.interp(T_NAT, band.doses, per_point)):.2f} hours per point, "
      f"band [{float(np.interp(T_NAT, band.doses, np.asarray(band.lower) / 100.0)):.2f}, "
      f"{float(np.interp(T_NAT, band.doses, np.asarray(band.upper) / 100.0)):.2f}]; "
      f"upper edge above zero anywhere on the grid: {bool((np.asarray(band.upper) > 0).any())}")
caption(fig, "Nearly two hours per point, and the band does not straddle zero anywhere "
             "on the range the sites ran. A negotiator trading a lower guarantee for a higher "
             "tax rate to save transfer cost is buying hours lost at this rate.")

## 5 · From the families enrolled to the families a national plan would reach

The experiments enrolled husband-wife families under 150 % of the poverty line in
eleven cities and two rural states. A national plan reaches every such family, a
quarter of them rural. The estimand notebook 1 declared for the national population
differs from the realized one in exactly one facet, and `transfer_to` says what
reading one as the other assumes.

In [ ]:
national = W.national_estimand(population=W.eligible_population(national=True), name="hours_under_national_plan_us")
plan = contrast.transfer_to(national)
print("transfer:", plan.status, "| differing facets:", plan.differing)
table([[a.name, a.facet, a.state, a.statement[:100]] for a in plan.assumptions],
      headers=("assumption", "facet", "state", "statement"))
for line in plan.ledger_lines:
    print("ledger:", line.kind, "—", line.statement)

`downgraded` under S-admissibility: the effect transports if whatever differs between
the enrolled and the national population is not something the plan's effect depends
on, once the right things are held fixed. That is a graph question, and the selection
diagram answers it. Let region — urban or rural — be the thing that differs. If region
affects hours directly, the effect transports by adjusting for region, and the rural
experiment supplies the rural stratum. If the populations differ in the *taste for
work* itself — the thing the survey could not measure — the verdict's route is
`trivial`: the only formula that identifies the national effect uses national data,
which is to say a national experiment.

In [ ]:
regional = CausalGraph.from_edges(
    "region -> hours, region -> schooling, taste -> hours, taste -> schooling, "
    "tax_rate -> hours, guarantee -> hours",
    unmeasured=["taste"], selection=["region"], name="region_differs",
)
through_taste = CausalGraph.from_edges(
    "region -> hours, taste -> hours, taste -> schooling, tax_rate -> hours, guarantee -> hours",
    unmeasured=["taste"], selection=["taste"], name="taste_differs",
)
for graph in (regional, through_taste):
    v = transport_verdict(graph, "guarantee", "hours")
    print(f"{graph.name:20s} {v.verdict.status:11s} route {v.route}"
          + (f" | adjust for {v.s_admissible_set}" if v.s_admissible_set else ""))
    for a in v.verdict.assumptions:
        print(f"    {a.name} ({a.state}): {a.statement[:100]}")
causal_graph(selection_diagram(through_taste), height=320)

## 6 · The decision, and what would change it

The Treasury's rule from notebook 1, applied to the realized contrast. Then the value
of information *after* the experiments, which is what says whether a fifth one is
worth running — and the sensitivity of the recommendation to the one number in it
that came from a memorandum rather than from data.

In [ ]:
decision = DecisionSpec(name="enact_national_plan", threshold=-W.HOURS_THRESHOLD,
                        value_per_outcome_unit=W.VALUE_PER_HOUR * W.N_ELIGIBLE, numeraire="USD")
posterior_mean, posterior_sd = answer.summary.mean, answer.summary.sd
before = evpi_gaussian(decision, -138.4, 80.9)  # notebook 1's prior
after = evpi_gaussian(decision, posterior_mean, posterior_sd)
print(f"hours change under the plan on the table: {posterior_mean:.0f} ± {posterior_sd:.0f}; "
      f"P(the plan is affordable) = {float(1 - __import__('scipy').stats.norm.cdf((decision.threshold - posterior_mean) / posterior_sd)):.4f}")
print(f"EVPI before the experiments: {before / 1e6:,.0f} million USD/yr; after: {after / 1e6:,.1f} million")
fiscal = (-posterior_mean) * W.VALUE_PER_HOUR * W.N_ELIGIBLE
transfer_cost = W.plan_cost(G_NAT, T_NAT) * W.N_ELIGIBLE
print(f"\nthe plan's hours cost the Treasury {fiscal / 1e9:.2f} bn USD/yr on top of {transfer_cost / 1e9:.1f} bn of transfers")

timing = time_to_re_experiment(posterior_sd=posterior_sd, half_life_periods=8.0, experiment_se=10.0, min_eig=0.5)
print(f"a fifth experiment pinning the contrast to ±10 hours is worth half a nat again in "
      f"{timing.periods:.0f} years, if the labour market drifts with an eight-year half-life")

In [ ]:
values = np.array([0.9, 1.35, 1.8, 2.7, 3.6, 5.4])
thresholds = W.HOURS_THRESHOLD * W.VALUE_PER_HOUR / values  # the same fiscal budget, at each price per hour
rows, labels, probs = [], [], []
for v, thr in zip(values, thresholds):
    p = float(1 - __import__("scipy").stats.norm.cdf((-thr - posterior_mean) / posterior_sd))
    rows.append([f"{v:.2f}", f"{thr:.0f}", f"{p:.3f}", "enact" if p > 0.5 else "hold"])
    labels.append(f"{v:.2f} USD/hour"); probs.append(p)
table(rows, headers=("value of an hour (USD)", "hours the same budget affords", "P(plan is affordable)", "decision"),
      title="the recommendation, as the memorandum's price of an hour moves")
fig = compare(
    labels, probs, highlight=f"{W.VALUE_PER_HOUR:.2f} USD/hour", value_fmt="{:.2f}",
    title="Where the recommendation changes its mind",
    subtitle="posterior probability the plan on the table is affordable, as the price of an hour moves",
    x_title="P(hours change is above the threshold)",
)
caption(fig, "The recommendation survives a doubling of the memorandum's price and flips at "
             "three times it. That is the argument the Treasury actually needs to have — about "
             "what an hour is worth — and it is a different argument from the one about how "
             "many hours the plan costs, which the experiments have settled.")

## 7 · The ledger

Every number above carries how it was made. The realized contrast's ledger, the
transfer's, and the fit's provenance are what a reader six months from now checks.

In [ ]:
print("realized contrast:", answer.estimand_name, "| status", answer.status, "| draws", answer.n_draws)
print("  producer hash", answer.producer_hash[:16], "| identification", answer.identification.status if answer.identification else "—")
print("  detail:", {k: v for k, v in answer.detail.items() if k in ("level", "basis", "n_units")})
print("\nfit provenance:", {k: linear.provenance[k] for k in ("spec_hash", "panel_hash", "backend", "draws", "seed")})
print("\nthe transfer to the national population:")
show(plan.ledger_lines[0])

## What notebook 5 established

1. **The data refuses a curvature and the refusal is typed.** A quadratic in the tax
   rate comes back `Unverified` with the direction the posterior is flat in; the
   linear surface with an interaction is what three sites' plans support.
2. **The plan on the table costs about 125 hours a year per family head**, with a
   90 % interval inside the Treasury's threshold of 150, and the truth of the synthetic
   world inside the interval. A 70 % tax rate does not clear the threshold at any
   guarantee.
3. **Each further point of tax rate costs nearly two hours**, and the band on
   that marginal effect does not straddle zero on the range the sites ran.
4. **Carrying the number to the national population is a transfer**, licensed by
   S-admissibility, which the selection diagram makes precise: it holds if region
   affects hours directly and adjusting for region is possible, and it fails if region
   moves the taste for work nobody measured.
5. **The experiments settled the question they were asked.** The value of perfect
   information fell from 270 million dollars a year to a tenth of a million; what
   would change the recommendation is not more hours data but a tripling of the
   memorandum's price of an hour, and the case study ends where that argument begins.